In [ ]:
import pandas as pd
import numpy as np
import json
from sentence_transformers import SentenceTransformer
import hdbscan
from sklearn.preprocessing import StandardScaler
from scipy import sparse
from sklearn.preprocessing import normalize
import re
import os
import math
import torch
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from sklearn.preprocessing import OneHotEncoder, RobustScaler, normalize
from scipy.sparse import hstack, csr_matrix
import torch
from sklearn.manifold import TSNE

# Load Data

In [ ]:
def load_dataframe(data_path='data.json', schema_path='schema.json'):
    # Load data
    df = pd.read_json(data_path, orient="records")
    
    # Load schema
    with open(schema_path) as f:
        schema = json.load(f)
    
    # Identify column types
    datetime_cols = [col for col, dtype in schema.items() if 'datetime64' in dtype]
    category_cols = [col for col, dtype in schema.items() if dtype == 'category']
    
    # Reparse datetimes
    for col in datetime_cols:
        df[col] = pd.to_datetime(df[col], utc='UTC' in schema[col])
    
    # Restore categories
    for col in category_cols:
        df[col] = df[col].astype('category')
    
    return df

file = load_dataframe(data_path="Data_Preprocessing.json", schema_path="Data_Preprocessing_schema.json")

# CLV Distribution (Histogram)

In [ ]:
fig = px.histogram(
    file,
    x="CLV",
    nbins=50,
    title="CLV Distribution",
    opacity=0.85
)
fig.show()

In [ ]:
clv_sorted = file['CLV'].dropna().sort_values(ascending=False)
cum_clv = clv_sorted.cumsum() / clv_sorted.sum()

fig = px.line(
    x=range(len(cum_clv)),
    y=cum_clv,
    labels={
        "x": "Customers (sorted by CLV)",
        "y": "Cumulative CLV Contribution"
    },
    title="Cumulative CLV Contribution Curve"
)

fig.add_hline(
    y=0.8,
    line_dash="dash",
    annotation_text="80% CLV threshold"
)

fig.show()

# Data-driven selection (CLV)

In [ ]:
customer_summary = (file.groupby("customer_id", as_index=False)
                    .agg(clv=("CLV", "max")))

customer_sorted = customer_summary.sort_values("clv", ascending=False).reset_index(drop=True)
cum_share = customer_sorted["clv"].cumsum() / customer_sorted["clv"].sum()

# Choose customers until we reach 80% of total CLV
cut_idx = (cum_share >= 0.80).idxmax()
top_customers = customer_sorted.iloc[:cut_idx + 1]
top_customer_ids = top_customers["customer_id"]

top_customer_rows = file[file["customer_id"].isin(top_customer_ids)]

pct = len(top_customers) / len(customer_summary) * 100

print(f"Selected customers for 80% cumulative CLV: {len(top_customers)} ({pct:.2f}%)")
print(f"Total bookings (rows) for these customers: {len(top_customer_rows)}")
print(f"CLV cutoff value: {top_customers['clv'].min():.2f}")

In [ ]:
check = file.groupby("customer_id")["CLV"].nunique()
print("Customers with >1 distinct CLV values:", (check > 1).sum())

In [ ]:
file = top_customer_rows.copy()

# HDBSCAN Clustering

## Feature Grouping

### Automatic suggested grouping (helper, not final)

In [ ]:
text_cols = []
categorical_low = []
categorical_high = []
numeric_cols = []
datetime_cols = []

for col in file.columns:
    dtype = file[col].dtype
    nunique = file[col].nunique(dropna=True)

    if pd.api.types.is_datetime64_any_dtype(dtype):
        datetime_cols.append(col)

    elif pd.api.types.is_numeric_dtype(dtype):
        numeric_cols.append(col)

    elif dtype == "object":
        if nunique < 30:
            categorical_low.append(col)
        elif nunique < 300:
            categorical_high.append(col)
        else:
            text_cols.append(col)

print("Text (embedding candidates):", text_cols)
print("Categorical (one-hot):", categorical_low)
print("Categorical (frequency):", categorical_high)
print("Numeric:", numeric_cols)
print("Datetime:", datetime_cols)

### Datetime (convert + engineer)

In [ ]:
date_cols = ["customer_registration", "customer_last_contact"]

for c in date_cols:
    if c in file.columns:
        file[c] = pd.to_datetime(file[c], errors="coerce")

In [ ]:
ref_date = file["ordered_at"].max()

In [ ]:
file["days_since_order"] = (ref_date - file["ordered_at"]).dt.days
file["customer_tenure_days"] = (ref_date - file["customer_registration"]).dt.days
file["days_since_last_contact"] = (ref_date - file["customer_last_contact"]).dt.days

In [ ]:
file = file.drop(columns=[
    "ordered_at",
    "customer_registration",
    "customer_last_contact"
])

In [ ]:
file_clustering = file.drop(columns=["customer_id"])

### Final, correct feature grouping

#### Text embeddings

In [ ]:
embedding = [
    "products_description",
    "products_title",
    "products_product_data_rateplan_title"
]

#### One-hot encoding

In [ ]:
one_hot_columns = [
    "products_type",
    "products_product_data_mealplan_code",
    "customer_language",
    "customer_gender",
    "customer_age_group"
]

#### Frequency encoding

In [ ]:
frequency_columns = [
    "customer_address_city",
    "customer_address_country_iso",
    "advertising_source",
    "advertising_campaign",
    "products_product_data_rateplan_code"
]

#### Numeric (scaled)

In [ ]:
numerical_cols = [
    "total_amount",
    "products_pu_participants",
    "products_participants_adult_count",
    "products_participants_child_count",
    "products_participants_teenager_count",
    "products_participants_infant_count",
    "customer_main",
    "customer_vip",
    "customer_average_stay_time",
    "customer_stay_count",
    "customer_total_revenue",
    "booking_lead_time",
    "occupancy_duration",
    "total_orders",
    "total_revenue",
    "avg_order_value",
    "lifespan_years",
    "CLV",
    "days_since_last_stay",
    "has_previous_stay",
    "days_since_order",
    "customer_tenure_days",
    "days_since_last_contact"
]

## Feature Representation and Encoding

In [ ]:
df = file_clustering.copy()

# (Recommended) reduce long-tail dominance for money-like cols
for c in ["CLV", "total_revenue", "customer_total_revenue", "avg_order_value", "total_amount"]:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")  # ensure numeric
        df.loc[df[c] < 0, c] = np.nan                  # treat negatives as missing (or clip to 0)
        df[c] = np.log1p(df[c])                        # log transform
# ------------------------
# 1) Text embeddings
# ------------------------

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)
model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2", device=device)

embedding_vectors = []
for col in embedding:
    print(f"Embedding column: {col}")
    texts = df[col].fillna("").astype(str).tolist()
    vecs = model.encode(texts, show_progress_bar=True, batch_size=16)
    embedding_vectors.append(vecs)

embedding_matrix = np.hstack(embedding_vectors)
embedding_matrix = normalize(embedding_matrix)

# ------------------------
# 2) One-hot encoding
# ------------------------
onehot_encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=True)
X_onehot_input = df[one_hot_columns].astype("string").fillna("MISSING")
onehot_matrix = onehot_encoder.fit_transform(X_onehot_input)
# ------------------------
# 3) Frequency encoding
# ------------------------
freq_encoded = []

for col in frequency_columns:
    s = df[col].astype("string").fillna("MISSING")
    freq_map = s.value_counts().to_dict()
    encoded = s.map(freq_map).astype(float).values.reshape(-1, 1)
    freq_encoded.append(encoded)

freq_encoded_matrix = (
    np.hstack(freq_encoded)
    if freq_encoded
    else np.zeros((len(df), 0))
)
# ------------------------
# 4) Numeric scaling
# ------------------------
scaler = RobustScaler()
numerical_data = scaler.fit_transform(df[numerical_cols].fillna(0))

# ------------------------
# 5) Final sparse feature matrix
# ------------------------
X_full = hstack([
    csr_matrix(embedding_matrix),
    onehot_matrix,
    csr_matrix(freq_encoded_matrix),
    csr_matrix(numerical_data)
]).tocsr()

print("X_full shape:", X_full.shape)


## HDBSCAN

### Stage 1 (before): Model selection

In [ ]:
import numpy as np
import pandas as pd
import hdbscan
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler, normalize
from sklearn.metrics import silhouette_score, davies_bouldin_score

# X_full is your sparse feature matrix (5773, 1196)

# --- scale (sparse-safe) ---
scaler = StandardScaler(with_mean=False)
X_scaled = scaler.fit_transform(X_full)

# --- reduce dimensionality (sparse-safe) ---
svd = TruncatedSVD(n_components=150, random_state=42)
X_reduced = svd.fit_transform(X_scaled)

# --- normalize rows (cosine-like with euclidean) ---
X_reduced_norm = normalize(X_reduced, norm="l2", axis=1)

def evaluate(labels, clusterer, X):
    mask = labels != -1
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    noise_pct = (labels == -1).mean() * 100

    # HDBSCAN confidence + stability
    mean_prob = float(clusterer.probabilities_[mask].mean()) if mask.any() else np.nan
    mean_persist = float(np.mean(clusterer.cluster_persistence_)) if n_clusters > 0 else np.nan

    # internal validation (only on non-noise)
    sil = silhouette_score(X[mask], labels[mask], metric="euclidean") if n_clusters > 1 else np.nan
    db  = davies_bouldin_score(X[mask], labels[mask]) if n_clusters > 1 else np.nan

    return n_clusters, noise_pct, mean_prob, mean_persist, sil, db

results = []

min_cluster_sizes = [150, 200, 250, 300, 350, 400]
min_samples_list  = [5, 10, 15, 20]

for mcs in min_cluster_sizes:
    for ms in min_samples_list:
        clusterer = hdbscan.HDBSCAN(
            min_cluster_size=mcs,
            min_samples=ms,
            metric="euclidean",
            cluster_selection_method="eom"
        )
        labels = clusterer.fit_predict(X_reduced_norm)

        n_clusters, noise_pct, mean_prob, mean_persist, sil, db = evaluate(labels, clusterer, X_reduced_norm)

        results.append({
            "min_cluster_size": mcs,
            "min_samples": ms,
            "clusters": n_clusters,
            "noise_%": round(noise_pct, 2),
            "mean_membership_prob": round(mean_prob, 3) if mean_prob==mean_prob else np.nan,
            "mean_persistence": round(mean_persist, 3) if mean_persist==mean_persist else np.nan,
            "silhouette_no_noise": round(sil, 3) if sil==sil else np.nan,
            "davies_bouldin_no_noise": round(db, 3) if db==db else np.nan
        })

df_eval = pd.DataFrame(results).sort_values(
    by=["mean_persistence", "mean_membership_prob"],
    ascending=False
)

df_eval

### Stage 2 (after): Final validation of the chosen model (run AFTER choosing 200 & 15)

In [ ]:
import numpy as np
import pandas as pd
import hdbscan
from sklearn.metrics import silhouette_score, davies_bouldin_score

# 1) Fit final chosen model
clusterer_final = hdbscan.HDBSCAN(
    min_cluster_size=200,
    min_samples=15,
    metric="euclidean",
    cluster_selection_method="eom",
    prediction_data=True
)

labels = clusterer_final.fit_predict(X_reduced_norm)

# 2) Basic summary
n_clusters = len(set(labels) - {-1})
noise_pct = (labels == -1).mean() * 100

print("Final model summary")
print("---------------")
print("clusters:", n_clusters)
print("noise %:", round(noise_pct, 2))

# 3) Membership confidence (non-noise)
mask = labels != -1
mean_prob = float(clusterer_final.probabilities_[mask].mean()) if mask.any() else np.nan
print("mean membership probability:", round(mean_prob, 3))

# 4) Cluster stability (persistence)
mean_persist = float(np.mean(clusterer_final.cluster_persistence_)) if n_clusters > 0 else np.nan
print("mean persistence:", round(mean_persist, 3))
print("cluster persistence per cluster:", np.round(clusterer_final.cluster_persistence_, 3))

# 5) Internal validation metrics (only on non-noise points)
if n_clusters > 1 and mask.sum() > 10:
    sil = silhouette_score(X_reduced_norm[mask], labels[mask], metric="euclidean")
    db = davies_bouldin_score(X_reduced_norm[mask], labels[mask])
else:
    sil, db = np.nan, np.nan

print("silhouette (no-noise):", round(float(sil), 3) if sil==sil else sil)
print("davies-bouldin (no-noise):", round(float(db), 3) if db==db else db)

# 6) Cluster size distribution (very important for personas)
cluster_sizes = pd.Series(labels[labels != -1]).value_counts().sort_index()
print("\nCluster sizes (excluding noise):")
print(cluster_sizes)

# 7) Attach labels to df for later analysis
df_validated = df.copy()
df_validated["persona_cluster"] = labels


### Robustness validation

In [ ]:
from sklearn.metrics import adjusted_rand_score

def run_labels(mcs, ms):
    c = hdbscan.HDBSCAN(min_cluster_size=mcs, min_samples=ms, metric="euclidean", cluster_selection_method="eom")
    return c.fit_predict(X_reduced_norm)

labels_a = run_labels(200, 15)  # final
labels_b = run_labels(200, 10)  # small change
labels_c = run_labels(250, 15)  # small change

def ari_no_noise(x, y):
    mask = (x != -1) & (y != -1)
    return adjusted_rand_score(x[mask], y[mask]) if mask.sum() > 0 else np.nan

print("ARI (200,15) vs (200,10):", round(ari_no_noise(labels_a, labels_b), 3))
print("ARI (200,15) vs (250,15):", round(ari_no_noise(labels_a, labels_c), 3))


###  Post-Clustering Noise Removal

In [ ]:
len(df), len(labels)

In [ ]:
df = df.copy()
df["persona_cluster"] = labels

In [ ]:
df_clean = df[df['persona_cluster'] != -1]

In [ ]:
print("Rows before:", len(df))
print("Rows after removing noise:", len(df_clean))
print("Dropped noise %:", 100 * (1 - len(df_clean) / len(df)))

In [ ]:
plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['font.size'] = 10

# These come from the standard HDBSCAN cell you just ran
unique_clusters = sorted([c for c in np.unique(labels) if c != -1])
cluster_sizes = pd.Series(labels[labels != -1]).value_counts().sort_index()
persistence = clusterer.cluster_persistence_

In [ ]:
# Figure 4 - Persistence
plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['font.size'] = 10

unique_clusters = list(range(9))
persistence = [0.087, 0.054, 0.649, 0.693, 0.054, 0.068, 0.172, 0.592, 0.625]

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(unique_clusters, persistence, color='#4a7ba6', edgecolor='black', linewidth=0.5)
mean_pers = np.mean(persistence)
ax.axhline(mean_pers, color='red', linestyle='--', linewidth=1, label=f'Mean = {mean_pers:.3f}')
ax.set_xlabel('Cluster ID'); ax.set_ylabel('Persistence')
ax.set_xticks(unique_clusters); ax.set_ylim(0, max(persistence)*1.15)
ax.legend(frameon=False)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('figure_4_cluster_persistence.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Figure 5 - Sizes
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(cluster_sizes.index, cluster_sizes.values, color='#6b8e76', edgecolor='black', linewidth=0.5)
for i, v in zip(cluster_sizes.index, cluster_sizes.values):
    ax.text(i, v+15, str(v), ha='center', fontsize=9)
ax.set_xlabel('Cluster ID'); ax.set_ylabel('Number of bookings')
ax.set_xticks(cluster_sizes.index); ax.set_ylim(0, max(cluster_sizes.values)*1.12)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('figure_5_cluster_size_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Figure 6 - t-SNE (use whatever variable holds the 150-dim SVD+L2 matrix)
mask = labels != -1
X_no_noise = X_reduced[mask]   # ← rename X_reduced if your variable is different
labels_no_noise = labels[mask]

print('Running t-SNE...')
tsne = TSNE(n_components=2, perplexity=30, random_state=42, init='pca', learning_rate='auto')
X_tsne = tsne.fit_transform(X_no_noise)

fig, ax = plt.subplots(figsize=(8, 6))
colors = plt.cm.tab10(np.linspace(0, 1, len(unique_clusters)))
for i, cid in enumerate(unique_clusters):
    pts = X_tsne[labels_no_noise == cid]
    ax.scatter(pts[:,0], pts[:,1], c=[colors[i]], label=f'Cluster {cid}', s=8, alpha=0.7, edgecolors='none')
ax.set_xlabel('t-SNE dimension 1'); ax.set_ylabel('t-SNE dimension 2')
ax.legend(frameon=False, loc='best', fontsize=9, markerscale=2)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('figure_6_tsne_projection.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Compute medians per cluster for the features in Table 5
summary = df_clean.groupby('persona_cluster').agg(
    median_stay_nights=('occupancy_duration', 'median'),
    median_lead_time_days=('booking_lead_time', 'median'),
    median_clv_eur=('CLV', 'median'),
    median_party_size=('products_pu_participants', 'median'),
).round(1)

# Repeat-visit share: share of bookings where has_previous_stay == 1
repeat_share = df_clean.groupby('persona_cluster')['has_previous_stay'].mean().round(3) * 100
summary['repeat_visit_share_%'] = repeat_share

print(summary)
summary.to_csv('table_5_cluster_features.csv')

In [ ]:
summary['median_clv_eur_real'] = (np.exp(summary['median_clv_eur']) - 1).round(0)
print(summary[['median_clv_eur', 'median_clv_eur_real']])

In [ ]:
plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['font.size'] = 10

# Five dimensions to compare
dimensions = ['Stay duration', 'Lead time', 'CLV', 'Party size', 'Repeat share']
clusters_to_show = [3, 4, 8]

# Build the data matrix from your summary dataframe
raw_values = {
    cid: [
        summary.loc[cid, 'median_stay_nights'],
        summary.loc[cid, 'median_lead_time_days'],
        summary.loc[cid, 'median_clv_eur_real'],
        summary.loc[cid, 'median_party_size'],
        summary.loc[cid, 'repeat_visit_share_%'],
    ]
    for cid in clusters_to_show
}

# Normalise each dimension to 0-1 across the three selected clusters
all_vals = np.array(list(raw_values.values()))
mins = all_vals.min(axis=0)
maxs = all_vals.max(axis=0)
ranges = np.where(maxs - mins == 0, 1, maxs - mins)
normalised = {cid: (np.array(v) - mins) / ranges for cid, v in raw_values.items()}

# Set up radar plot
n_dims = len(dimensions)
angles = np.linspace(0, 2 * np.pi, n_dims, endpoint=False).tolist()
angles += angles[:1]  # close the loop

fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))
colors = ['#4a7ba6', '#c97a3a', '#6b8e76']
labels = {3: 'Cluster 3 (Long-stay couple)', 4: 'Cluster 4 (Family planner)', 8: 'Cluster 8 (Week-stay couple)'}

for i, cid in enumerate(clusters_to_show):
    values = normalised[cid].tolist()
    values += values[:1]  # close the loop
    ax.plot(angles, values, color=colors[i], linewidth=2, label=labels[cid])
    ax.fill(angles, values, color=colors[i], alpha=0.15)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(dimensions)
ax.set_ylim(0, 1)
ax.set_yticks([0.25, 0.5, 0.75, 1.0])
ax.set_yticklabels(['0.25', '0.50', '0.75', '1.00'], fontsize=8)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), frameon=False, fontsize=9)
ax.tick_params(axis='x', pad=20)
plt.tight_layout()
plt.savefig('figure_7_radar_chart.png', dpi=300, bbox_inches='tight')
plt.show()

## Adaptive HDBSCAN

In [ ]:
# =========================
# Adaptive HDBSCAN on X_full
# Focus: clustering only
# =========================

import numpy as np
from sklearn.preprocessing import StandardScaler, normalize
from sklearn.decomposition import TruncatedSVD
from sklearn.cluster import AgglomerativeClustering
import hdbscan


# -------------------------
# 0) Sanity check
# -------------------------
print("X_full shape:", X_full.shape)


# -------------------------
# 1) Scale (sparse-safe), reduce (SVD), normalize
# -------------------------
scaler = StandardScaler(with_mean=False)
X_scaled = scaler.fit_transform(X_full)

svd_k = min(150, X_scaled.shape[1] - 1)
svd = TruncatedSVD(n_components=svd_k, random_state=42)
X_red = svd.fit_transform(X_scaled)

X_red_norm = normalize(X_red, norm="l2", axis=1)

print("Reduced shape:", X_red_norm.shape)


# -------------------------
# 2) Adaptive HDBSCAN search
# -------------------------
def run_adaptive_hdbscan(
    X,
    target_min=6,
    target_max=9,
    trials_max=120
):
    n = X.shape[0]

    def _score(k, noise):
        # Prefer solutions in target range with low noise
        if target_min <= k <= target_max:
            return noise
        # Penalize distance from target range + some noise penalty
        dist = (target_min - k) if k < target_min else (k - target_max)
        return dist + 0.5 * noise

    # Grid (biased toward fewer clusters)
    frac_grid = [0.02, 0.03, 0.04, 0.05, 0.06, 0.08, 0.10]
    min_sizes = sorted({max(5, int(n * f)) for f in frac_grid} | {25, 50, 75, 100, 150, 200})
    min_samples_list = [1, 5, 10, 15]
    eps_list = [0.0, 0.01, 0.02, 0.05, 0.08, 0.10]
    methods = ["eom", "leaf"]

    tried = 0
    best = None

    for mcs in min_sizes:
        for ms in min_samples_list:
            for eps in eps_list:
                for method in methods:
                    if tried >= trials_max:
                        break
                    tried += 1

                    cl = hdbscan.HDBSCAN(
                        min_cluster_size=mcs,
                        min_samples=ms,
                        cluster_selection_epsilon=eps,
                        metric="euclidean",
                        cluster_selection_method=method,
                        prediction_data=False,
                    )
                    labels = cl.fit_predict(X)
                    k = len(set(labels) - {-1})
                    noise = float(np.mean(labels == -1))
                    sc = _score(k, noise)

                    cand = (sc, {"min_cluster_size": mcs, "min_samples": ms,
                                 "epsilon": eps, "method": method},
                            labels, noise, k)

                    if best is None or sc < best[0]:
                        best = cand

                    # If we hit the target range, we can return early
                    if target_min <= k <= target_max:
                        return best

    return best


best = run_adaptive_hdbscan(X_red_norm, target_min=6, target_max=9, trials_max=120)

if best is None:
    raise RuntimeError("Adaptive HDBSCAN search failed.")

score, config, labels, noise_ratio, k = best
print(f"[BEST] config={config}")
print(f"[BEST] clusters(excl -1)={k}, noise={noise_ratio:.2%}, score={score:.3f}")


# -------------------------
# 3) Post-merge if too many clusters
# -------------------------
def merge_clusters_to_target(X, labels, target_min=6, target_max=9):
    uniq = sorted([c for c in set(labels) if c != -1])
    k = len(uniq)
    if k <= target_max:
        return labels

    target_k = max(target_min, min(target_max, k))
    if target_k == k:
        return labels

    # Centroids of clusters
    centroids = np.vstack([X[labels == cid].mean(axis=0) for cid in uniq])

    agg = AgglomerativeClustering(n_clusters=target_k, metric="euclidean", linkage="ward")
    meta = agg.fit_predict(centroids)

    cid_to_idx = {c: i for i, c in enumerate(uniq)}
    new_labels = labels.copy()
    for i, c in enumerate(labels):
        if c == -1:
            continue
        new_labels[i] = int(meta[cid_to_idx[c]])

    return new_labels


labels_final = merge_clusters_to_target(X_red_norm, labels, target_min=6, target_max=9)
final_k = len(set(labels_final) - {-1})
print(f"[FINAL] clusters(excl -1)={final_k}")


# -------------------------
# 4) Drop noise and attach labels back to df
# -------------------------
mask = labels_final != -1
print(f"Rows kept (non-noise): {mask.sum()} / {len(mask)}")

df_Adaptive_HDBSCAN = df.loc[mask].copy()
df_Adaptive_HDBSCAN["persona_cluster"] = labels_final[mask]

display(df_Adaptive_HDBSCAN["persona_cluster"].value_counts().sort_index()) 

In [ ]:
from sklearn.metrics import adjusted_rand_score
import hdbscan
import numpy as np

# -------------------------
# Helper: run HDBSCAN with fixed params
# -------------------------
def run_hdbscan(X, min_cluster_size, min_samples, epsilon=0.0, method="eom"):
    cl = hdbscan.HDBSCAN(
        min_cluster_size=min_cluster_size,
        min_samples=min_samples,
        cluster_selection_epsilon=epsilon,
        metric="euclidean",
        cluster_selection_method=method,
        prediction_data=False
    )
    return cl.fit_predict(X)


# -------------------------
# BASELINE = FINAL CONFIG
# -------------------------
labels_base = run_hdbscan(
    X_red_norm,
    min_cluster_size=75,
    min_samples=1,
    epsilon=0.0,
    method="eom"
)

# -------------------------
# SMALL PARAMETER PERTURBATIONS
# -------------------------
configs = [
    (60, 1, 0.0),
    (75, 5, 0.0),
    (90, 1, 0.0),
    (75, 1, 0.02),
]

results = []

for mcs, ms, eps in configs:
    labels_alt = run_hdbscan(X_red_norm, mcs, ms, eps)

    # compare ONLY common non-noise points
    mask = (labels_base != -1) & (labels_alt != -1)

    ari = adjusted_rand_score(
        labels_base[mask],
        labels_alt[mask]
    ) if mask.sum() > 0 else np.nan

    results.append({
        "min_cluster_size": mcs,
        "min_samples": ms,
        "epsilon": eps,
        "points_compared": mask.sum(),
        "ARI": ari
    })

results

In [ ]:
df_clean.to_csv("high_clv_bookings_with_persona_clusters.csv", index=False)